# Getting Swiss supermarkets using the overpass Web API

## Overpass turbo query to get supermarkets in the city of Zürich

In [ ]:
import json
import time

import folium
import requests
from pandas import json_normalize

# Overpass API endpoints to try, in order (some public mirrors are
# unreliable / rate-limited, so we fall back through a list)
urls = [
    'https://overpass.osm.ch/api/interpreter',
    'https://overpass-api.de/api/interpreter',
]

# Overpass turbo query (supermarkets in the city of Zürich)
query = """
        [out:json][timeout:60];
        area["name"="Zürich"]["admin_level"="8"]->.searchArea;
        node["shop"="supermarket"](area.searchArea);
        out;"""

# Web API request with retries (the Overpass servers are often busy)
max_attempts = 6
wait_time = 10  # seconds between attempts
data = None

for attempt in range(1, max_attempts + 1):
    url = urls[(attempt - 1) % len(urls)]
    try:
        r = requests.post(url, data={'data': query}, timeout=90)
        r.raise_for_status()
        data = r.json()['elements']
        print(
            f'Attempt {attempt} ({url}): success, '
            f'{len(data)} elements received.'
        )
        break
    except (requests.exceptions.RequestException, ValueError, KeyError) as e:
        print(f'Attempt {attempt} ({url}) failed: {e}')
        if attempt < max_attempts:
            print(f'Waiting {wait_time} seconds before retrying ...')
            time.sleep(wait_time)
            wait_time *= 2  # exponential backoff
        else:
            raise RuntimeError(
                'Overpass API not available after several attempts.')

# Save data to file
with open('supermarkets.json', 'w') as json_file:
    json.dump(data, json_file)

# Store data in data frame
df = json_normalize(data)
df.head(5)

## Plot supermarkets on map

In [ ]:
# Subset of supermarkets by brand
locations = df[["lat", "lon", "tags.brand", "tags.shop"]
               ].loc[df["tags.brand"].isin(['Coop', 'Migros'])]
print(locations.head(5))

# Subsets
locations_coop = df[["lat", "lon", "tags.brand",
                     "tags.shop"]].loc[df["tags.brand"].isin(['Coop'])]
locations_migros = df[["lat", "lon", "tags.brand",
                       "tags.shop"]].loc[df["tags.brand"].isin(['Migros'])]

# Marker symbols
marker_base_url = (
    'https://raw.githubusercontent.com/pointhi/'
    'leaflet-color-markers/master/img'
)
url_01 = f'{marker_base_url}/marker-icon-green.png'
url_02 = f'{marker_base_url}/marker-icon-orange.png'

# Create map
map = folium.Map(location=[locations.lat.mean(),
                           locations.lon.mean()],
                 zoom_start=12,
                 control_scale=True)

# Add lat/lon of Coop supermarkets
for i in range(0, len(locations_coop)):
    folium.Marker(
        location=(
            locations_coop.iloc[i]['lat'],
            locations_coop.iloc[i]['lon']),
        popup=locations_coop.iloc[i]['tags.brand'],
        icon=folium.features.CustomIcon(
            url_01,
            icon_size=(
                16,
                28))).add_to(map)

# Add lat/lon of Migros supermarkets
for i in range(0, len(locations_migros)):
    folium.Marker(
        location=(
            locations_migros.iloc[i]['lat'],
            locations_migros.iloc[i]['lon']),
        popup=locations_migros.iloc[i]['tags.brand'],
        icon=folium.features.CustomIcon(
            url_02,
            icon_size=(
                16,
                28))).add_to(map)

# Plot map
map

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')